In [1]:
import requests
import numpy as np
import pandas as pd
import pickle

# Load test data to send real network flows to the API
# This demonstrates the API working on actual CICIDS 2017 data
with open('../models/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_test = data['X_test']
y_test = data['y_test']
le = data['label_encoder']

print(f"Loaded {len(X_test):,} real network flows for testing")
print(f"Will send these to the API and compare predictions vs actual labels")

Loaded 44,253 real network flows for testing
Will send these to the API and compare predictions vs actual labels


In [2]:
BASE_URL = "http://localhost:5000"

# Send 20 real network flows from the test set to the API
print("Sending 20 real network flows to the API...\n")

results = []
correct = 0

for i in range(20):
    features = X_test[i].tolist()
    actual = le.inverse_transform([y_test[i]])[0]

    r = requests.post(f"{BASE_URL}/predict",
                      json={"features": features})
    result = r.json()

    predicted = result['prediction']
    confidence = result['confidence_pct']
    is_correct = actual == predicted
    if is_correct:
        correct += 1

    icon = "✓" if is_correct else "✗"
    threat_icon = "🚨" if result['threat_detected'] else "  "
    print(f"{threat_icon} {icon} Flow {i+1:2d}: "
          f"Actual={actual:8s} | "
          f"Predicted={predicted:8s} | "
          f"Confidence={confidence:6.2f}%")

    results.append({
        'actual': actual,
        'predicted': predicted,
        'correct': is_correct,
        'confidence': confidence
    })

print(f"\nAccuracy on 20 real flows: {correct}/20 ({correct/20*100:.0f}%)")

Sending 20 real network flows to the API...

   ✓ Flow  1: Actual=Benign   | Predicted=Benign   | Confidence= 95.00%
   ✗ Flow  2: Actual=DDoS     | Predicted=Benign   | Confidence= 95.00%
   ✓ Flow  3: Actual=Benign   | Predicted=Benign   | Confidence= 95.00%
   ✗ Flow  4: Actual=DDoS     | Predicted=Benign   | Confidence= 94.00%
   ✗ Flow  5: Actual=DDoS     | Predicted=Benign   | Confidence= 94.00%
   ✓ Flow  6: Actual=Benign   | Predicted=Benign   | Confidence= 90.00%
   ✗ Flow  7: Actual=DDoS     | Predicted=Benign   | Confidence= 92.00%
   ✓ Flow  8: Actual=Benign   | Predicted=Benign   | Confidence= 96.00%
   ✓ Flow  9: Actual=Benign   | Predicted=Benign   | Confidence= 93.00%
   ✗ Flow 10: Actual=DDoS     | Predicted=Benign   | Confidence= 93.00%
   ✗ Flow 11: Actual=DDoS     | Predicted=Benign   | Confidence= 95.00%
   ✗ Flow 12: Actual=DDoS     | Predicted=Benign   | Confidence= 94.00%
   ✓ Flow 13: Actual=Benign   | Predicted=Benign   | Confidence= 91.00%
   ✗ Flow 14: Actua

In [3]:
# Pull stats from the API
r = requests.get(f"{BASE_URL}/stats")
stats = r.json()

print("Live API Statistics:")
print(f"  Total flows analyzed: {stats['total_analyzed']}")
print(f"  Threats detected:     {stats['threats_detected']}")
print(f"  Detection rate:       {stats['detection_rate_pct']}%")
print(f"  Model:                {stats['model_type']}")
print(f"  Model accuracy:       {stats['model_accuracy']}")

Live API Statistics:
  Total flows analyzed: 30
  Threats detected:     0
  Detection rate:       0.0%
  Model:                Random Forest (100 trees)
  Model accuracy:       99.98%
